# Phase 3 — Self-Training (CBST) + Fine-tuning (Colab)

Picks up from the Phase 2 pretrained 9-class checkpoint and runs:
1. **Pseudo-labeling** of BaDLAD-unlabeled with the pretrained model
2. **CBST** class-balanced thresholds (rare classes not swamped)
3. **Self-training rounds** on {real + pseudo}
4. **Fine-tune on IndicDLP** (re-headed to IndicDLP's ontology) — *added next*

Logic lives in `src/finetuning/self_training.py`; these cells only orchestrate.
**Runtime:** T4 is fine for the smoke test (Cell 4). Switch to **A100** for the full run (Cell 5), off-peak.

## Cell 0 — Bootstrap (mount Drive, install deps)

In [1]:
# Runtime -> Change runtime type -> A100 (full run) or T4 (smoke test) -> Save FIRST
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
PROJECT_ROOT = Path('/content/drive/MyDrive/doclayout-yolo-indic')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))


subprocess.run(['pip','install','-q','ultralytics','huggingface_hub','pyyaml',
                'pycocotools','kagglehub','tqdm'], check=True)
subprocess.run(['pip','install','-q',
                'git+https://github.com/opendatalab/DocLayout-YOLO.git'], check=True)

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
print('PROJECT_ROOT:', PROJECT_ROOT)

Mounted at /content/drive
PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-40GB
PROJECT_ROOT: /content/drive/MyDrive/doclayout-yolo-indic


## Cell 1 — Sync code from GitHub (plain files, no zip)
Clones the repo, auto-detects `src/` (works whether it's at the repo root or inside a
wrapper dir like `doclayout-yolo-indic/`), and copies it to `PROJECT_ROOT/src` on Drive so
`config.py`'s `__file__`-anchored paths resolve to Drive (persistent outputs/checkpoints).
**Prereq:** commit `src/` as plain files (drop the zip) and add
`src/finetuning/self_training.py` + `__init__.py`.

In [2]:
import subprocess, shutil
from pathlib import Path

GITHUB_URL    = 'https://github.com/vigneshpalanivelr/mtech-project-aiml.git'
GITHUB_BRANCH = 'main'
REPO_LOCAL    = Path('/content/_repo')

shutil.rmtree(REPO_LOCAL, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','-b',GITHUB_BRANCH,
                GITHUB_URL, str(REPO_LOCAL)], check=True)

# Find src/ wherever it lives: repo root, or one level down (wrapper dir).
candidates = [REPO_LOCAL/'src'] + sorted(REPO_LOCAL.glob('*/src'))
SRC = next((c for c in candidates if (c/'config.py').exists()), None)
assert SRC, f'Could not find src/config.py under {REPO_LOCAL}'
BASE = SRC.parent
print('Found project at:', BASE)

# Copy code (not docs) to PROJECT_ROOT so REPO_ROOT = parents[1] -> Drive.
for item in ['src','tests','requirements.txt','README.md']:
    s = BASE/item; d = PROJECT_ROOT/item
    if s.is_dir():   shutil.rmtree(d, ignore_errors=True); shutil.copytree(s, d)
    elif s.exists(): shutil.copy(s, d)

assert (PROJECT_ROOT/'src'/'finetuning'/'self_training.py').exists(), \
    'Commit src/finetuning/self_training.py + __init__.py to the repo first!'
print('Code synced (plain files) ->', PROJECT_ROOT/'src')

Found project at: /content/_repo/doclayout-yolo-indic
Code synced (plain files) -> /content/drive/MyDrive/doclayout-yolo-indic/src


## Cell 2 — Download BaDLAD (Kaggle: reasat/badlad-train)
4 classes: text-box, paragraph, image, table. Re-pulled each session (don't persist to Drive).
**Prereq:** add `KAGGLE_USERNAME` + `KAGGLE_KEY` to Colab Secrets (left key panel).

In [3]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_API_TOKEN')

import kagglehub
BADLAD_PATH = kagglehub.dataset_download('reasat/badlad-train')
print('BaDLAD cached at:', BADLAD_PATH)
for root, dirs, files in os.walk(BADLAD_PATH):
    print(root, '->', len(files), 'files')
    if root != BADLAD_PATH: break
# Set this to the folder that actually holds the .jpg/.png pages:
BADLAD_IMAGES = BADLAD_PATH  # adjust after inspecting the printout above

100%|██████████| 18.7G/18.7G [09:45<00:00, 34.3MB/s]

Extracting files...


BaDLAD cached at: /root/.cache/kagglehub/datasets/reasat/badlad-train/versions/2
/root/.cache/kagglehub/datasets/reasat/badlad-train/versions/2 -> 1 files
/root/.cache/kagglehub/datasets/reasat/badlad-train/versions/2/badlad_train -> 0 files


### Cell 2.1 — Build labeled BaDLAD set

In [4]:
from pathlib import Path
import kagglehub, json

BADLAD_PATH = Path(kagglehub.dataset_download('reasat/badlad-train'))
img_dir = BADLAD_PATH / 'badlad_train'

direct    = list(img_dir.glob('*.png'))      # flat: images directly in the folder
recursive = list(img_dir.rglob('*.png'))     # nested: images anywhere below it
coco = json.load(open(BADLAD_PATH / 'badlad-train-coco.json'))

print(f"images directly in badlad_train/ : {len(direct)}")
print(f"images recursively (any subfolder): {len(recursive)}")
print(f"images the COCO json expects       : {len(coco['images'])}")

# is one of the 'missing' files actually present, just somewhere else?
sample = 'c707dfd3-3655-4ae5-8e78-e658437eba77.png'
print(f"'{sample}' found at: {list(img_dir.rglob(sample))}")

images directly in badlad_train/ : 0
images recursively (any subfolder): 20365
images the COCO json expects       : 20365
'c707dfd3-3655-4ae5-8e78-e658437eba77.png' found at: [PosixPath('/root/.cache/kagglehub/datasets/reasat/badlad-train/versions/2/badlad_train/badlad_train/c707dfd3-3655-4ae5-8e78-e658437eba77.png')]


In [5]:
from src.finetuning.data_prep import prepare_badlad
yaml_path, exclude_file = prepare_badlad(
    coco_json  = f"{BADLAD_PATH}/badlad-train-coco.json",
    images_dir = f"{BADLAD_PATH}/badlad_train",
    drive_out  = PROJECT_ROOT/"data"/"badlad_9class",
    local_root = "/content/badlad_9class",
)
print("labeled yaml:", yaml_path)

03:10:54 | INFO    | doclayout_indic.data_prep | Reusing Drive cache (train 8000 imgs / val 2000 imgs) -> local
03:20:43 | INFO    | doclayout_indic.data_prep | Wrote /content/drive/MyDrive/doclayout-yolo-indic/data/badlad_9class/badlad_9class.yaml
labeled yaml: /content/drive/MyDrive/doclayout-yolo-indic/data/badlad_9class/badlad_9class.yaml


## Cell 3 — Stage IndicDLP from your private HF repo
Pull only the splits you need. Upload it once from local first (see chat instructions).

In [6]:
from pathlib import Path
from huggingface_hub import snapshot_download, login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))
INDICDLP_DIR = Path('/content/IndicDLP')
if INDICDLP_DIR.exists() and any(INDICDLP_DIR.rglob('*.json')):
    print('IndicDLP already staged at', INDICDLP_DIR)
else:
    snapshot_download(repo_id='VigneshPR/IndicDLP', repo_type='dataset',
                      local_dir=str(INDICDLP_DIR),
                      allow_patterns=['train/*','val/*','*.json','*.yaml'])
    print('IndicDLP staged ->', INDICDLP_DIR)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

IndicDLP staged -> /content/IndicDLP


## Cell 4 — Smoke test: CBST pseudo-labeling on ~50 images (T4 OK)
Verifies the *novel* part — inference + class-balanced thresholds + label writing — fast, no full train.

In [7]:
%cd {PROJECT_ROOT}
from src.finetuning.self_training import (load_detector, collect_predictions,
    compute_cbst_thresholds, filter_with_thresholds, write_pseudo_labels)

PRETRAINED = PROJECT_ROOT/'output'/'checkpoints'/'doclayout_yolo_indic_pretrained.pt'

model, names = load_detector(PRETRAINED)
recs = collect_predictions(model, BADLAD_IMAGES, device=0, limit=50)
thr  = compute_cbst_thresholds(recs, num_classes=len(names), portion=0.20)
kept, stats = filter_with_thresholds(recs, thr)

print('CBST thresholds :', {names[k]: round(v,3) for k,v in thr.items()})
print('kept/total/class:', {names[int(k)]: v for k,v in stats.items()})
write_pseudo_labels(kept, '/content/pseudo_smoke', names)
print(f'OK -- wrote {len(kept)} pseudo-label files. CBST pipeline works.')

/content/drive/MyDrive/doclayout-yolo-indic
03:21:04 | INFO    | doclayout_indic.self_training | Loaded detector doclayout_yolo_indic_pretrained.pt | 9 classes -> ['text_body', 'headline', 'table', 'figure', 'caption', 'advertisement', 'sidebar', 'pull_quote', 'decorative_frame']
03:21:04 | INFO    | doclayout_indic.self_training | Pseudo-label inference over 50 images from /root/.cache/kagglehub/datasets/reasat/badlad-train/versions/2
03:21:10 | INFO    | doclayout_indic.self_training | Collected 278 detections over 48 images (0 skipped)
CBST thresholds : {'text_body': 0.987, 'headline': 0.875, 'table': 1.01, 'figure': 0.726, 'caption': 0.868, 'advertisement': 1.01, 'sidebar': 0.798, 'pull_quote': 0.626, 'decorative_frame': 0.722}
kept/total/class: {'text_body': {'kept': 32, 'total': 161}, 'headline': {'kept': 12, 'total': 61}, 'figure': {'kept': 1, 'total': 1}, 'caption': {'kept': 8, 'total': 43}, 'sidebar': {'kept': 1, 'total': 4}, 'pull_quote': {'kept': 1, 'total': 1}, 'decorative_

## Cell 5 — Full self-training (A100, off-peak)
Two CBST rounds, resumable. `LABELED=None` runs pseudo-only now; once `data_prep.py`
builds the BaDLAD 9-class yaml, point `LABELED` at it for the supervised half.

In [8]:
import os
os.environ['TQDM_MININTERVAL'] = '30'   # redraw the bar at most every 30s

%cd {PROJECT_ROOT}
from src.finetuning.self_training import run_self_training

PRETRAINED = PROJECT_ROOT/'output'/'checkpoints'/'doclayout_yolo_indic_pretrained.pt'
LABELED    = None  # or PROJECT_ROOT/'data'/'raw'/'BaDLAD'/'badlad_9class.yaml'

#final_best = run_self_training(
#    pretrained   = PRETRAINED,
#    unlabeled_dir= BADLAD_IMAGES,
#    labeled_yaml = LABELED,
#    work_dir     = PROJECT_ROOT/'output'/'self_training',
#    num_rounds   = 2,
#    device       = 0)


#final_best = run_self_training(
#    pretrained    = PRETRAINED,
#    unlabeled_dir = BADLAD_IMAGES,
#    labeled_yaml  = None,
#    work_dir      = PROJECT_ROOT/'output'/'self_training',
#    num_rounds    = 2,
#    device        = 0,
#    infer_limit   = 300)     # round 2 re-infers on only 300 images -> ~2 min

final_best = run_self_training(
    pretrained         = PRETRAINED,
    unlabeled_dir      = BADLAD_IMAGES,
    labeled_yaml       = str(yaml_path),
    work_dir           = PROJECT_ROOT/'output'/'self_training_real',   # NEW folder
    num_rounds         = 2,
    device             = 0,
    exclude_stems_file = str(exclude_file),
)
print('Final self-training checkpoint:', final_best)

/content/drive/MyDrive/doclayout-yolo-indic
03:21:10 | INFO    | doclayout_indic.self_training | Loaded 10000 exclude stems (labeled+val) from /content/drive/MyDrive/doclayout-yolo-indic/data/badlad_9class/exclude_stems.json
03:21:10 | INFO    | doclayout_indic.self_training | === Self-training round 1/2 (portion=20%) ===
03:21:12 | INFO    | doclayout_indic.self_training | Round 1 already trained -> /content/drive/MyDrive/doclayout-yolo-indic/output/self_training_real/round_1/train/weights/best.pt (skipping)
03:21:12 | INFO    | doclayout_indic.self_training | === Self-training round 2/2 (portion=30%) ===
03:21:12 | INFO    | doclayout_indic.self_training | Round 2 already trained -> /content/drive/MyDrive/doclayout-yolo-indic/output/self_training_real/round_2/train/weights/best.pt (skipping)
03:21:12 | INFO    | doclayout_indic.self_training | Self-training complete. Final checkpoint: /content/drive/MyDrive/doclayout-yolo-indic/output/self_training_real/round_2/train/weights/best.pt


## Next
- `src/finetuning/data_prep.py` — BaDLAD COCO->YOLO 9-class remap + IndicDLP yaml
- `src/finetuning/train_finetuning.py` — re-head to IndicDLP's 42 classes + fine-tune (final model)
- `src/finetuning/ablation.py` — the 3-5 ablation runs

(`src/utils/ontology.py` already exists: pretrain 9-class head, then **replace the head**
with IndicDLP's 42 classes at fine-tuning. Run its `inspect_indicdlp_categories()` first.)

In [9]:
from pathlib import Path

def _n(d, ext): return len(list(Path(d).glob(ext))) if Path(d).exists() else 0

checks = {
    'round_1 best.pt': (PROJECT_ROOT/'output'/'self_training_real'/'round_1'/'train'/'weights'/'best.pt').exists(),
    'round_2 best.pt': (PROJECT_ROOT/'output'/'self_training_real'/'round_2'/'train'/'weights'/'best.pt').exists(),
    'badlad cache val imgs':   _n(PROJECT_ROOT/'data'/'badlad_9class'/'cache'/'images'/'val',   '*.jpg') >= 1900,
    'badlad cache train imgs': _n(PROJECT_ROOT/'data'/'badlad_9class'/'cache'/'images'/'train', '*.jpg') >= 7800,
}
for k, ok in checks.items():
    print(('OK  ' if ok else 'MISSING  ') + k)

if all(checks.values()):
    print('\nAll critical artifacts on Drive — safe to release the runtime.')
    from google.colab import runtime
    runtime.unassign()
else:
    print('\nDO NOT unassign yet — something above is missing.')

OK  round_1 best.pt
OK  round_2 best.pt
OK  badlad cache val imgs
OK  badlad cache train imgs

All critical artifacts on Drive — safe to release the runtime.
